# 04 · Validación del pipeline
Comprueba que el pipeline se ejecutó, que las tablas Gold cumplen claves e integridad, que los KPIs se calculan y que
la base SQL quedó cargada.

In [1]:
import sys, warnings
from pathlib import Path
RAIZ = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)
GOLD, SILVER, CTL = RAIZ/'data'/'gold', RAIZ/'data'/'silver', RAIZ/'data'/'ctl'

## 1. Pruebas automatizadas (pytest)

In [2]:
import subprocess
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', str(RAIZ/'tests')], capture_output=True, text=True, cwd=RAIZ)
print(r.stdout[-1500:])

.........................                                                [100%]
25 passed in 5.26s



## 2. Base de datos SQL: tablas, filas e integridad referencial

In [3]:
import sqlite3
con = sqlite3.connect(RAIZ/'database'/'etl_corea.sqlite')
tablas = pd.read_sql("select name from sqlite_master where type='table' order by name", con)
filas = [(t, con.execute(f'select count(*) from "{t}"').fetchone()[0]) for t in tablas.name]
display(pd.DataFrame(filas, columns=['tabla','filas']))
print('Violaciones de claves foráneas:', con.execute('PRAGMA foreign_key_check').fetchall())

,tabla,filas
0,bronze_manifest,28
1,bronze_registros,20813
2,ctl_conteos,146
3,ctl_log_cargas,33
4,ctl_log_cargas_historico,68
5,ctl_rechazos,82
6,ctl_validaciones,318
7,gold_dataset_nacional_anual,103
8,gold_dataset_regional_anual,915
9,gold_dim_edad,30


Violaciones de claves foráneas: []


## 3. Consultas analíticas de ejemplo (como las usaría Power BI)

In [4]:
q = '''select t.periodo, f.anio, round(f.valor,1) as dep_vejez, f.tipo_dato
from gold_fact_indicador_historico f join gold_dim_tiempo t on t.anio=f.anio
where f.cod_indicador='DEP_VEJEZ' and f.cod_territorio='00' and f.cod_sexo='T' and f.anio in (2000,2010,2022)
union all
select t.periodo, f.anio, round(f.valor,1), f.tipo_dato from gold_fact_indicador_proyeccion f join gold_dim_tiempo t on t.anio=f.anio
where f.cod_indicador='DEP_VEJEZ' and f.cod_territorio='00' and f.cod_sexo='T' and f.cod_escenario='medio'
  and f.edicion_proyeccion='KOSTAT 2022-2072' and f.anio in (2030,2050,2072) order by 2'''
pd.read_sql(q, con)

,periodo,anio,dep_vejez,tipo_dato
0,Histórico,2000,10.1,calculado
1,Histórico,2010,14.8,calculado
2,Histórico,2022,24.4,calculado
3,Proyección,2030,38.0,calculado
4,Proyección,2050,77.3,calculado
5,Proyección,2072,104.2,calculado


In [5]:
pd.read_sql('''select s.nombre as supuesto, f.anio, round(sum(f.fuerza_laboral_potencial)/1e6,2) as millones
from gold_fact_fuerza_laboral_escenario f join gold_dim_supuesto s using(cod_supuesto)
where f.cod_escenario='medio' and f.anio in (2025,2050) group by 1,2 order by 1,2''', con)

,supuesto,anio,millones
0,A · Participación constante,2025,30.05
1,A · Participación constante,2050,26.39
2,B · Tendencia 2015-2025 amortiguada,2025,30.05
3,B · Tendencia 2015-2025 amortiguada,2050,28.40
4,C · Cierre del 50 % de la brecha de género,2025,30.05
5,C · Cierre del 50 % de la brecha de género,2050,28.14


## 4. KPIs de calidad por dataset

In [6]:
pd.read_parquet(GOLD/'kpi_calidad_dataset.parquet').sort_values('tasa_validos_pct').head(10)

,dataset,registros_evaluados,registros_validos,registros_rechazados,rechazos_trazados,tasa_validos_pct,tasa_rechazo_pct,rechazos_trazados_pct,cumple_validos,cumple_rechazo
17,censo_registros,1330,1260,70,70.0,94.737,5.263,100.0,False,False
19,eaps_sido,4068,4059,9,9.0,99.779,0.221,100.0,True,True
29,vitales_sido_mensual,1371,1368,3,3.0,99.781,0.219,100.0,True,True
0,OECD/fecundidad/NACIMIENTOS,176,176,0,0.0,100.000,0.000,100.0,True,True
4,OECD/fuerza_laboral/TASA_DESEMPLEO,181,181,0,0.0,100.000,0.000,100.0,True,True
1,OECD/fecundidad/TFR,173,173,0,0.0,100.000,0.000,100.0,True,True
2,OECD/fuerza_laboral/OCUPADOS,181,181,0,0.0,100.000,0.000,100.0,True,True
3,OECD/fuerza_laboral/POB_ACTIVA,181,181,0,0.0,100.000,0.000,100.0,True,True
8,WB/SP.DYN.LE00.IN,225,225,0,0.0,100.000,0.000,100.0,True,True
9,WB/SP.DYN.TFRT.IN,225,225,0,0.0,100.000,0.000,100.0,True,True


In [7]:
o = pd.read_parquet(GOLD/'kpi_okr.parquet'); proc = o[o.tipo_kpi=='proceso']
print(f"KPIs de proceso que cumplen: {int(proc.cumple.sum())} de {len(proc)}")
proc[['kr','kpi','valor','meta','cumple']]

KPIs de proceso que cumplen: 0 de 0


,kr,kpi,valor,meta,cumple
